In [16]:
from datetime import datetime, timedelta

In [17]:
from clearwater_modules_v2.processes.temperature import Temperature

In [18]:
from clearwater_data.variables.registry import VariableRegistry
from clearwater_data.variables.float import FloatVariable

In [19]:
#Instantiate the Variable Registry
# This is were you will define data the process will access
variable_registry = VariableRegistry()



In [20]:
#To add data to the registry you need a variable
water_temp = FloatVariable(20)
#water_temp = FloatVariable(40)
surf_area = FloatVariable(1)
vol = FloatVariable(1)

#air_temp_c = FloatVariable(20.0) #was 20.0 in v1 example
air_temp_c = FloatVariable(30.0)
q_solar = FloatVariable(400.0) #was 400.0 in v1 example
#sed_temp_c = FloatVariable(5.0)

eair_mb = FloatVariable(1.0) # v1 example uses 1.0. This is the atmospheric vapor pressure.
#This value looks like a low value; a google search shows typical values around 10mb

pressure_mb = FloatVariable(1013.0)
cloudiness_frct = FloatVariable(0.1)
wind_spd = FloatVariable(3.0)

sed_temp_c = FloatVariable(5.0) #was 5.0 in v1 example
sed_thick_m = FloatVariable(0.1) #looks like the the thickness was 0.1m in the v1 example

#define wind function parameters as floats
wind_a_userInput = 0.3
wind_b_userInput = 1.5
wind_c_userInput = 1.0
#wind_kh_kw = FloatVariable(1.0) #this is the air diffusivity ratio in v1 example
#this parameter is used in v2 temperature process and it defaults to 1.0 so no need to define it here

sediment_density_userInput = 1600 #kg/m3 (1600 kg/m3 was used in v1 example);
#default value in v2 temperature process is NOW 1600 kg/m3

sediment_specific_heat_userInput = 1673.0 #J/kg/K was used in v1 example
#default value in v2 temperature process is 1000.0 J/kg/K

sediment_diffusivity_userInput = 0.0432 #m^2/s #used in v1 example
#default value in v2 temperature process is 0.0061 m^2/s

#define time step for the process
time_step_userInput = timedelta(seconds=1) #also tested with 300sec (5min)


In [21]:
#Now you can register this
variable_registry.register("water_temperature", water_temp)
variable_registry.register("wetted_surface_area", surf_area)
variable_registry.register("volume", vol)
variable_registry.register("cloudiness", cloudiness_frct)
variable_registry.register("air_temperature", air_temp_c)
variable_registry.register("solar_radiation", q_solar)
variable_registry.register("wind_speed", wind_spd)
variable_registry.register("atmospheric_pressure", pressure_mb)
variable_registry.register("atmospheric_vapor_pressure", eair_mb)
variable_registry.register("sediment_temperature", sed_temp_c)
variable_registry.register("sediment_thickness", sed_thick_m)


In [22]:
float(variable_registry.get("air_temperature"))

30.0

In [23]:
float(variable_registry.get("cloudiness"))

0.1

In [24]:
process = Temperature(
    wind_a=wind_a_userInput,
    wind_b=wind_b_userInput,
    wind_c=wind_c_userInput,
    sediment_density=sediment_density_userInput,
    sediment_specific_heat=sediment_specific_heat_userInput,
    sediment_diffusivity=sediment_diffusivity_userInput,
    time_step=time_step_userInput
)

In [25]:
date_time = datetime(2026, 1, 1, 0, 0, 0)
date_time

datetime.datetime(2026, 1, 1, 0, 0)

In [26]:
#variable_registry.get("water_temperature")


In [27]:
#process.run(date_time, variable_registry)

In [28]:
#variable_registry.get("water_temperature")


In [29]:
temp_time_series = []

for _ in range(1):
    print(f'Iteration: {_}')
    temp_at_start = float(variable_registry.get("water_temperature"))
    process.run(date_time, variable_registry)
    temp_at_end = float(variable_registry.get("water_temperature"))
    temp_pair = (temp_at_start, temp_at_end)
    print(f'    Temperature Pair: {temp_pair}')
    temp_time_series.append(temp_pair)

#temp_time_series

Iteration: 0
    Richardson Number: 0.054606644322854586
      gravity: -9.806
      density_air: 1.1628315081145268
      density_air_sat: 1.1919710624251105
      wind_speed: 3.0
Case 1 == False
Case 2 == False
Case 3 == False
Case 4 == True
    Richardson Function: 0.43183446784380736
    Latent heat terms:
      atmospheric pressure: 1013.0
      latent_heat_vaporization: 1800595.4616
        water_temperature: 20.0
      water_density: 998.2066838317065
      wind_function: 2.0728054456502753e-06
        wind_speed: 3.0
      saturation_vapor_pressure: 23.371004503124823
      atmospheric_vapor_pressure: 1.0
    Longwave down terms:
      cloudiness_term: 1.0017
        cloudiness_frac: 0.1
      emissivity_air: 0.8611590850720001
      stefan_boltzmann: 5.67e-08
      air_temp_term: 8446710189.10741
        air_temp_k: 303.16
    Temperature Pair: (20.0, 19.999994011759732)


D:\Clearwater\ClearWater-modules\src\clearwater_modules_v2\processes\temperature.py:638: RuntimeWarning: invalid value encountered in scalar power
  (1.0 - 22.0 * richardson_number) ** 0.80,


In [ ]:
from clearwater_modules.tsm import EnergyBudget
from clearwater_modules.tsm.constants import (
    Meteorological,
    Temperature,
)

In [ ]:
def initial_tsm_state() -> dict[str, float]:
    """Return initial state values for the model."""
    return {
        'water_temp_c': 20.0,
        'surface_area': 1.0,
        'volume': 1.0,
    }

In [ ]:
def time_steps() -> int:
    return 1

In [ ]:
def default_meteo_params() -> Meteorological:
    """Returns default meteorological static variable values for the model.

    NOTE: As of now (11/17/2023) these match the built in defaults, but are 
    copied here to allow for easy modification of the defaults in the future.

    Returns a typed dictionary, with string keys and float values.
    """
    return Meteorological(
        air_temp_c=20.0,
        q_solar=400.0,
        sed_temp_c=5.0,
        eair_mb=1.0,
        pressure_mb=1013.0,
        cloudiness=0.1,
        wind_speed=3.0,
        wind_a=0.3,
        wind_b=1.5,
        wind_c=1.0,
        wind_kh_kw=1.0,
    )

In [ ]:
check_default_metPara = default_meteo_params()
check_default_metPara

In [ ]:
def default_temp_params() -> Temperature:
    """Returns default temperature static variable values for the model.

    NOTE: As of now (11/17/2023) these match the built in defaults, but are 
    copied here to allow for easy modification of the defaults in the future.

    Returns a typed dictionary, with string keys and float or bool values.
    """
    return Temperature(
        stefan_boltzmann=5.67e-8,
        cp_air=1005.0,
        emissivity_water=0.97,
        gravity=-9.806,
        a0=6984.505294,
        a1=-188.903931,
        a2=2.133357675,
        a3=-1.288580973E-2,
        a4=4.393587233E-5,
        a5=-8.023923082E-8,
        a6=6.136820929E-11,
        pb=1600.0,
        cps=1673.0,
        h2=0.1,
        alphas=0.0432,
        richardson_option=True,
        dt=1/86400, # 1 second
    )

In [ ]:
def get_energy_budget_instance(
    time_steps,
    initial_tsm_state,
    default_meteo_params,
    default_temp_params,
    use_sed_temp: bool = True,
) -> EnergyBudget:
    """Return an instance of the TSM class."""
    return EnergyBudget(
        time_steps=time_steps,
        initial_state_values=initial_tsm_state,
        meteo_parameters=default_meteo_params,
        temp_parameters=default_temp_params,
        use_sed_temp=use_sed_temp,
        time_dim='tsm_time_step',
    )


In [ ]:
def tolerance() -> float:
    """Controls the precision of the pytest.approx() function."""
    return 0.0000001

In [ ]:
def test_defaults(
    time_steps,
    initial_tsm_state,
    default_meteo_params,
    default_temp_params,
    tolerance,
) -> None:
    """Test the model with default parameters."""
    # alter parameters as necessary

    # instantiate the model
    tsm: EnergyBudget = get_energy_budget_instance(
        time_steps=time_steps,
        initial_tsm_state=initial_tsm_state,
        default_meteo_params=default_meteo_params,
        default_temp_params=default_temp_params,
    )

    # Run the model
    tsm.increment_timestep()
    water_temp_c = tsm.dataset.isel(
        tsm_time_step=-1).water_temp_c.values.item()
    return water_temp_c

In [ ]:
test_defaults(
    time_steps(),
    initial_tsm_state(),
    default_meteo_params(),
    default_temp_params(),
    tolerance(),
)

In [ ]:
def default_temperature_dict() -> dict[str, float]:
    """Return default values for the pytest model."""
    return {
        'water_temperature': 20.0,
        'wetted_surface_area': 1.0,
        'volume': 1.0,
        'cloudiness': 0.1,
        'air_temperature': 20.0,
        'solar_radiation': 400.0,
        'wind_speed': 3.0,
        'atmospheric_pressure': 1013.0,
        'atmospheric_vapor_pressure': 1.0,
        'sediment_temperature': 5.0,
        'sediment_thickness': 0.1,
        'wind_a': 0.3,
        'wind_b': 1.5,
        'wind_c': 1.0,
        'sediment_density': 1600.0,
        'sediment_specific_heat': 1673.0,
        'air_diffusivity_ratio': 1.0,
        'sediment_diffusivity': 0.0432,
        'time_step': 1.0,
        'water_temperature_compare': 19.9999461,
        }

In [ ]:
def parametrize_changes_dict() -> dict[str, float]:
    """Return default values for the pytest model."""
    return {
        'test_defaults': {
            'water_temperature': 20.0,
            'water_temperature_compare': 19.9999461,
        },

        'test_changed_water_temp': {
            'water_temperature': 40.0,
            'water_temperature_compare': 39.99939598,
        },

    }

In [ ]:
defaultTemp_dict = default_temperature_dict()
defaultTemp_dict

In [ ]:
paraChanges_dict = parametrize_changes_dict()
paraChanges_dict

In [ ]:
import copy

parametrize_id_list = []
parametrize_list = []

for primary_key, primary_value in paraChanges_dict.items():
    print(f"Primary Key: {primary_key}")
    parametrize_id_list.append(primary_key)
    defaultTemp_dict_updated = copy.deepcopy(defaultTemp_dict)
    for secondary_key, secondary_value in primary_value.items():
        print(f"    Secondary Key: {secondary_key}, Value: {secondary_value}")
        defaultTemp_dict_updated[secondary_key] = secondary_value
    parametrize_test_tuple = tuple(defaultTemp_dict_updated.values())
    parametrize_list.append(parametrize_test_tuple)

In [ ]:
parametrize_id_list

In [ ]:
parametrize_list